Part b

In [4]:
import numpy as np
import pandas as pd
from collections import Counter

def calculate_probabilities(values):
    count = Counter(values)
    total = len(values)
    return {key: value / total for key, value in count.items()}

def calculate_joint_probabilities(x, y):
    total = len(x)
    joint_counts = Counter(zip(x, y))
    return {key: value / total for key, value in joint_counts.items()}

def mutual_information_manual(x, y):
    # Marginal probabilities
    px = calculate_probabilities(x)
    py = calculate_probabilities(y)
    # Joint probabilities
    pxy = calculate_joint_probabilities(x, y)
    mi = 0
    for (x_val, y_val), pxy_val in pxy.items():
        px_val = px[x_val]
        py_val = py[y_val]
        mi += pxy_val * np.log(pxy_val / (px_val * py_val))
    return mi

def mutual_information(X, y):
    mi_scores = []
    for i in range(X.shape[1]):
        mi = mutual_information_manual(X[:, i], y)
        mi_scores.append(mi)
    return mi_scores

def correlation_coefficient(X, y):
    correlations = []
    for i in range(X.shape[1]):
        corr = np.corrcoef(X[:, i], y)[0, 1]
        correlations.append(abs(corr))  # Take absolute value
    return correlations

def main(data_path, labels_path):
    X = pd.read_csv(data_path).values
    y = pd.read_csv(labels_path).values.flatten()
    corr_scores = correlation_coefficient(X, y)
    print("Correlation Coefficients for each feature:")
    for i, score in enumerate(corr_scores):
        print(f"Feature {i}: {score:.4f}")
    mi_scores = mutual_information(X, y)
    print("\nMutual Information for each feature:")
    for i, score in enumerate(mi_scores):
        print(f"Feature {i}: {score:.4f}")

data_path = "F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Pattern Recognition/Assignments/HW4/Answer/Input/P5/breast_data.csv"
labels_path = "F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Pattern Recognition/Assignments/HW4/Answer/Input/P5/breast_labels.csv"
main(data_path, labels_path)

Correlation Coefficients for each feature:
Feature 0: 0.7294
Feature 1: 0.4221
Feature 2: 0.7419
Feature 3: 0.7084
Feature 4: 0.3563
Feature 5: 0.5956
Feature 6: 0.6956
Feature 7: 0.7764
Feature 8: 0.3273
Feature 9: 0.0181
Feature 10: 0.5654
Feature 11: 0.0070
Feature 12: 0.5544
Feature 13: 0.5463
Feature 14: 0.0666
Feature 15: 0.2909
Feature 16: 0.2526
Feature 17: 0.4073
Feature 18: 0.0092
Feature 19: 0.0761
Feature 20: 0.7757
Feature 21: 0.4614
Feature 22: 0.7825
Feature 23: 0.7329
Feature 24: 0.4197
Feature 25: 0.5895
Feature 26: 0.6583
Feature 27: 0.7932
Feature 28: 0.4134
Feature 29: 0.3210

Mutual Information for each feature:
Feature 0: 0.5960
Feature 1: 0.5786
Feature 2: 0.6417
Feature 3: 0.6427
Feature 4: 0.5372
Feature 5: 0.6295
Feature 6: 0.6475
Feature 7: 0.6524
Feature 8: 0.5086
Feature 9: 0.5789
Feature 10: 0.6466
Feature 11: 0.5984
Feature 12: 0.6451
Feature 13: 0.6408
Feature 14: 0.6475
Feature 15: 0.6393
Feature 16: 0.6427
Feature 17: 0.5944
Feature 18: 0.5663
Feature 

Part c

In [8]:
def sequential_forward_selection(X, y, objective_func, k):
    selected_features = []
    remaining_features = list(range(X.shape[1])) 
    for _ in range(k):
        best_score = -np.inf
        best_feature = None
        for feature in remaining_features:
            current_features = selected_features + [feature]
            scores = objective_func(X[:, current_features], y)
            score = np.mean(scores)   
            if score > best_score:
                best_score = score
                best_feature = feature
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
    return selected_features

def bidirectional_selection(X, y, objective_func, k):
    selected_features = []
    remaining_features = list(range(X.shape[1]))
    for _ in range(k):
        best_score = -np.inf
        best_feature = None
        for feature in remaining_features:
            current_features = selected_features + [feature]
            scores = objective_func(X[:, current_features], y)
            score = np.mean(scores)Z
            if score > best_score:
                best_score = score
                best_feature = feature
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)        
        worst_score = np.inf
        worst_feature = None    
        for feature in selected_features:
            current_features = [f for f in selected_features if f != feature]
            scores = objective_func(X[:, current_features], y)
            score = np.mean(scores)       
            if score < worst_score:
                worst_score = score
                worst_feature = feature     
        if worst_feature is not None:
            selected_features.remove(worst_feature)
    return selected_features

def sequential_floating_selection(X, y, objective_func, k):
    selected_features = []
    remaining_features = list(range(X.shape[1]))
    for _ in range(k):
        best_score = -np.inf
        best_feature = None  
        for feature in remaining_features:
            current_features = selected_features + [feature]
            scores = objective_func(X[:, current_features], y)
            score = np.mean(scores)   
            if score > best_score:
                best_score = score
                best_feature = feature
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)        
        worst_score = np.inf
        worst_feature = None
        for feature in selected_features:
            current_features = [f for f in selected_features if f != feature]
            scores = objective_func(X[:, current_features], y)
            score = np.mean(scores)
            if score < worst_score:
                worst_score = score
                worst_feature = feature
        if worst_feature is not None:
            selected_features.remove(worst_feature)
    return selected_features

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_regression_manual(X, y, lr=0.01, epochs=1000):
    weights = np.zeros(X.shape[1])
    bias = 0
    for _ in range(epochs):
        linear_model = np.dot(X, weights) + bias
        predictions = sigmoid(linear_model)
        errors = predictions - y
        weight_gradient = np.dot(X.T, errors) / len(y)
        bias_gradient = np.sum(errors) / len(y)
        weights -= lr * weight_gradient
        bias -= lr * bias_gradient
    return weights, bias

def predict_manual(X, weights, bias):
    linear_model = np.dot(X, weights) + bias
    probabilities = sigmoid(linear_model)
    return np.where(probabilities >= 0.5, 1, 0)

def accuracy_manual(y_true, y_pred):
    correct_predictions = np.sum(y_true == y_pred)
    return correct_predictions / len(y_true)

def evaluate_features(X, y, selected_features):
    X_subset = X[:, selected_features]
    weights, bias = logistic_regression_manual(X_subset, y)
    y_pred = predict_manual(X_subset, weights, bias)
    return accuracy_manual(y, y_pred)

def main(data_path, labels_path, k=5):
    X = pd.read_csv(data_path).values
    y = pd.read_csv(labels_path).values.flatten()
    objective_functions = [
        (correlation_coefficient, "Correlation Coefficient"),
        (mutual_information, "Mutual Information")
    ]
    
    results = []
    for obj_func, name in objective_functions:
        for method, method_name in [
            (sequential_forward_selection, "SFS"),
            (bidirectional_selection, "BDS"),
            (sequential_floating_selection, "SFS Floating")
        ]:
            selected_features = method(X, y, obj_func, k)
            accuracy = evaluate_features(X, y, selected_features)
            results.append({
                "Objective Function": name,
                "Method": method_name,
                "Selected Features": selected_features,
                "Accuracy": accuracy
            })
    results_df = pd.DataFrame(results)
    results_df.to_excel("F:/Aidin/Universities/Amirkabir (Master)/Lessons/Term 2/Pattern Recognition/Assignments/HW4/Answer/Output/P5/feature_selection_results.xlsx", index=False)
    return results_df

results = main(data_path, labels_path)
results


C:\Users\TamirLand\AppData\Roaming\Python\Python39\site-packages\numpy\core\fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\TamirLand\AppData\Roaming\Python\Python39\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\TamirLand\AppData\Local\Temp\ipykernel_8404\481293675.py:92: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


,Objective Function,Method,Selected Features,Accuracy
0,Correlation Coefficient,SFS,"[27, 22, 7, 20, 2]",0.690141
1,Correlation Coefficient,BDS,[2],0.371479
2,Correlation Coefficient,SFS Floating,[2],0.371479
3,Mutual Information,SFS,"[7, 23, 6, 14, 10]",0.371479
4,Mutual Information,BDS,[10],0.640845
5,Mutual Information,SFS Floating,[10],0.640845
